# Session 2: Building the Modeling Table
**Emission-trajectory ML project. Dr. Khawar Naeem, QTTSC, Qatar University.**

This notebook teaches what `src/build_features.py` does and why. The script is the deliverable; this notebook is the classroom. Work through it top to bottom, then run the script itself from the terminal:

```
python src/build_features.py
```

**The big idea:** we build ONE flat table, once, containing the target, every feature, and the split labels. Every model in Sessions 3-5 reads this same table. Models may differ in algorithm, never in data. This is the ML analogue of the SQL staging table you built in your joins practice.

## 1. The target: shift(-1) within each country

A supervised learner needs (features, answer) pairs in the same row. Our answer is *next year's* co2, which lives one row below. `shift(-1)` pulls it up.

**The data structure that makes this work:** the table sorted by (country, year), then `groupby("country")`. The groupby is a wall: pandas performs the shift inside each country's block only, so Zimbabwe's last year gets `NaN` as its target instead of stealing Zambia's first year.

In [ ]:
import pandas as pd

raw = pd.read_csv("../data/raw/owid-co2-data.csv")
is_country = raw["iso_code"].notna() & ~raw["iso_code"].astype(str).str.startswith("OWID")
df = raw[is_country].sort_values(["country", "year"]).reset_index(drop=True)

# The one-line heart of the whole project:
df["target_co2_next"] = df.groupby("country")["co2"].shift(-1)

# Watch it work on Qatar: 2023's target must be 2024's co2 value, 125.812
df[df["country"] == "Qatar"][["year", "co2", "target_co2_next"]].tail(5)

## 2. The trap: shift is positional, not temporal

`shift(1)` means "the previous ROW," not "the previous YEAR." Those are the same thing only when every year is present consecutively. If a country's series had a gap (say 1991 missing), `shift(1)` on the 1992 row would silently grab 1990: a wrong lag with no error message.

The script's defense (in `_one_country`) is to reindex each country onto its full calendar range first, inserting explicit NaN rows for any missing year. On a gap-free series this changes nothing; on a gapped one it converts a silent bug into an honest NaN. Cheap insurance.

Run the cell below to see the mechanism on a toy series with a deliberate gap.

In [ ]:
toy = pd.DataFrame({"year": [1990, 1991, 1993, 1994], "co2": [10.0, 11.0, 13.0, 14.0]})  # 1992 missing!

naive_lag = toy["co2"].shift(1)                       # positional: 1993 gets 1991's value. WRONG lag.
fixed = toy.set_index("year").reindex(range(1990, 1995))
fixed["co2_lag1"] = fixed["co2"].shift(1)             # temporal: 1993 gets NaN. HONEST.

print("naive (positional):"); print(toy.assign(lag1=naive_lag).to_string(index=False))
print(); print("reindexed (temporal):"); print(fixed.reset_index().to_string(index=False))

## 3. Lags, rolling means, and the slope

Three feature families, all built from co2's own history, all legal because their windows END at year t:

- **Lags** (`shift(k)`): the level k years ago. Gives the model memory.
- **Rolling means** (`rolling(5).mean()`): at year t this covers t-4..t inclusive; smooths out one-off spikes. Note we do NOT need shift-before-roll here, because year t's own value is available at prediction time. Shift-before-roll is only for windows that must EXCLUDE year t.
- **Slope** (`(co2 - co2.shift(5)) / 5`): average yearly change over the last five years; the model's sense of direction. This is also exactly what the linear-trend baseline in Session 3 will use, which is deliberate.

And one honesty subtlety in the growth features: `pct_change`'s default forward-fills gaps before differencing, fabricating a growth rate across missing years. The script passes `fill_method=None` so a gap yields NaN, which is the truth.

In [ ]:
g = df[df["country"] == "Qatar"].set_index("year")["co2"]

demo = pd.DataFrame({
    "co2": g,
    "co2_lag1": g.shift(1),
    "co2_lag5": g.shift(5),
    "co2_roll5_mean": g.rolling(5).mean(),
    "co2_slope5": (g - g.shift(5)) / 5,
})
demo.tail(6)  # verify each column by mental arithmetic against the co2 column

## 4. Order of operations: features BEFORE filtering

The eligibility rules (>= 40 years of co2 history, population >= 1M) DELETE rows. If we filtered first, the deletions would punch holes in each country's series, and every shift afterward would be corrupted (section 2's trap, self-inflicted).

So the pipeline order is: load -> build target and features on complete series -> THEN filter rows -> label splits. Two more details worth owning:

- The 40-year rule uses each country's whole history to decide eligibility. That is a static design choice about which countries are modelable, not a feature, so it does not leak into any prediction.
- The plan said population > 1M "in the prediction year" (t+1), but t+1's population is not knowable at prediction time. The script measures it at year t instead, and this deviation is recorded. Small decisions, explicitly owned, are what make the repo defensible.

## 5. Splits keyed on the TARGET year

A row whose features describe 2014 predicts 2015. If we keyed splits on the feature year, that row would land in training while its answer comes from validation's first year. The vintage of the ANSWER is what the split protects, so `split` is assigned from `target_year = year + 1`:

| split | target years | meaning |
|---|---|---|
| train | ...-2014 | model fitting |
| val | 2015-2018 | model selection and tuning |
| test | 2019-2023 | touched ONCE, at the end (headline) |
| provisional_2024 | 2024 | side table only; newest OWID year is revisable |

In [ ]:
proc = pd.read_csv("../data/processed/modeling_table.csv")  # run src/build_features.py first
print(proc["split"].value_counts().reindex(["train", "val", "test", "provisional_2024"]).to_string())
print()
print(f"{proc['country'].nunique()} countries, feature years {proc['year'].min()}-{proc['year'].max()}")
print()
# every val row predicts 2015-2018, exactly 4 per country:
print("val rows per country all == 4:", (proc[proc['split']=='val'].groupby('country').size() == 4).all())

## 6. What the eligibility rules cost us

218 countries entered; 153 survived. Where did 65 go? Mostly the population >= 1M rule (microstates) and the wave-1 row-drop (countries missing cement/flaring series). The cell below produces the accounting; a version of this belongs in the README's data-limitations section, because "which countries does your model silently exclude" is exactly the question a careful reviewer asks.

In [ ]:
entered = set(df["country"].unique())
survived = set(proc["country"].unique())
excluded = sorted(entered - survived)
print(f"entered: {len(entered)}, survived: {len(survived)}, excluded: {len(excluded)}")
print(excluded)

## 7. Verification is part of the artifact

`build_features.py` refuses to save a table that fails its checks: Qatar's answer key, 200 random targets re-derived from the raw CSV, 200 random lags checked temporally, no duplicate country-years, no target beyond 2024. Assertions in the pipeline are the difference between "I think the table is right" and "the table cannot ship wrong."

## Check questions (close Session 2 by answering these)

1. Write, from memory, the pandas expression for a 5-year rolling mean of co2 that uses only information available at year t. Why is shift-before-roll NOT needed here, and when would it be?
2. What happens in each country's earliest rows when `co2_lag10` is created, and what does the pipeline do with those rows? What does each country's FINAL row lack, and why?
3. Qatar's 2023 row: state its target value, and which split it belongs to and why (two separate decisions are involved).
4. Why must features be built before eligibility filtering, and not after?
5. A colleague splits train/val/test on the feature year instead of the target year. Which specific rows are now mislabeled, and which direction does the resulting evaluation bias go?

**End-of-session protocol:** commit `src/build_features.py` and this notebook, update `CLAUDE.md` status and session log, record the exact next action (Session 3: baselines in `notebooks/02_baselines.ipynb` reading `data/processed/modeling_table.csv`).